# D6 — multi-head joint ablation (one compound at a time)

Spec (approved 2026-06-29): `docs/superpowers/specs/2026-06-29-multihead-lexical-ablation-design.md`.
Pre-registration: DECISIONS.md D6 entry — **commit before the first real forward pass**.
Claim under test, scoped: even the strongest lexical lead (screen reader) is
distributed across heads, not a localized circuit. Model: pythia-2.8b throughout.

**Upload to this Colab session:** this notebook + the `src/` folder (drag the folder
into the Files sidebar) + `results/binding/pythia/pythia-2.8b/pythia-2.8b-accessibility.csv`
at that same path, so Stage A reads the frozen sweep; out-of-sweep compounds like
stock_market run a fresh in-memory binding pass and say so in the ledger).

Flow: run Setup once → `sanity_check` once → edit Cell 1 (compound) → run Cell 2 →
repeat for screen_reader, alt_text, stock_market, semantic_html → zip and bring it
home. Model loads once per session; only the compound changes between visits.

## Setup

In [ ]:
# Cell 00: Colab Stuff
import os

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    # NOTE: no 'pip install --upgrade numpy' here — on current Colab images it
    # desyncs numpy from the preinstalled scipy's compiled ABI
    # (ImportError: _center from numpy._core.umath). D8's Cell 00 still carries
    # the line above its own removal note; fixed here, fix D8 when convenient.
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [1]:
# Cell 0: Project root + imports (expects the src/ FOLDER uploaded, not flat files)
import sys
from pathlib import Path
import torch

# Colab: files are in /content/ ; Local: notebook lives in notebooks/
if Path('/content/src').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / 'src' / 'd6_multihead_ablation.py').exists(), (
    'Upload the src/ folder next to this notebook (Files sidebar, drag the '
    'whole folder) — d6 needs binding, head_characterization, qk_ov, and '
    'd6_multihead_ablation together.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.d6_multihead_ablation import run_d6, sanity_check, COMPOUNDS
print(f"Project root: {PROJECT_ROOT}")
print("Compounds in the panel:", list(COMPOUNDS))

Project root: /Users/trishasalas/Repos/Research/tmlr
Compounds in the panel: ['screen_reader', 'alt_text', 'stock_market', 'semantic_html']


In [2]:
# Cell 0b: Load the model ONCE per session (single model, per spec)
from transformer_lens import HookedTransformer

if 'model' not in globals():
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model = HookedTransformer.from_pretrained('pythia-2.8b', device=device)
    print(f"Loaded pythia-2.8b on {device} "
          f"({model.cfg.n_layers} layers x {model.cfg.n_heads} heads)")
else:
    print("Model already loaded — reusing.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-2.8b into HookedTransformer
Loaded pythia-2.8b on mps (32 layers x 32 heads)


In [3]:
# Cell 0c: Sanity asserts (spec: identity KL==0; plural==singular on L29/H7;
# late-layer set moves KL). Run ONCE before the first real ablation.
sanity_check(model)

sanity_check PASSED — identity KL 0.0; L29/H7 singular==plural (1e-05); late-layer set (L30, 8 heads) KL=3e-06


## One compound per visit
Edit Cell 1, run Cell 2. Order: `screen_reader` (primary — also runs the
bicycle-wheel negative control) → `alt_text` → `stock_market` → `semantic_html`.
Cell 3 shows the ledger. Reruns replace a compound's rows, never duplicate.

In [13]:
# Cell 1: Compound name variable
# screen_reader | alt_text | stock_market | semantic_html
compound_name = "semantic_html"

In [14]:
# Cell 2: Run D6 for this compound
# Stage A earns the lexical set (deep + selective + not sink/structural; the
# excluded heads become the labeled positive-control tail), Stage B runs the
# cumulative-knockout curve, both ledgers upsert. Seal untouched, no generation.
curves = run_d6(model, PROJECT_ROOT, compound_name)
curves

Stage A [semantic_html]: 18 deep candidates (min_layer=10, source: frozen CSV)
  earned lexical set: 13 heads — L30H4, L27H27, L28H1, L27H13, L10H6, L27H9, L10H26, L14H12, L23H12, L28H28, L21H15, L23H22, L18H9
  Stage A table → /Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-candidate-heads.csv
Stage B [semantic_html]: cumulative ablation over 13 heads (13 lexical + 0 tail)
  Stage B curves → /Users/trishasalas/Repos/Research/tmlr/results/pythia/pythia-2.8b-multihead-ablation.csv

[semantic_html] peak KL — lexical set: 0.001471 | with structural tail: (no tail)
Flat lexical segment + rising tail = distributed result + working instrument, in one curve.


,compound,n_ablated,heads,newest_head,kl,segment,role
0,semantic_html,1,L30H4,L30H4,0.000005,lexical,robustness
1,semantic_html,2,L30H4; L27H27,L27H27,0.000010,lexical,robustness
2,semantic_html,3,L30H4; L27H27; L28H1,L28H1,0.000012,lexical,robustness
3,semantic_html,4,L30H4; L27H27; L28H1; L27H13,L27H13,0.000032,lexical,robustness
4,semantic_html,5,L30H4; L27H27; L28H1; L27H13; L10H6,L10H6,0.000352,lexical,robustness
5,semantic_html,6,L30H4; L27H27; L28H1; L27H13; L10H6; L27H9,L27H9,0.000386,lexical,robustness
6,semantic_html,7,L30H4; L27H27; L28H1; L27H13; L10H6; L27H9; L1...,L10H26,0.001122,lexical,robustness
7,semantic_html,8,L30H4; L27H27; L28H1; L27H13; L10H6; L27H9; L1...,L14H12,0.001272,lexical,robustness
8,semantic_html,9,L30H4; L27H27; L28H1; L27H13; L10H6; L27H9; L1...,L23H12,0.001236,lexical,robustness
9,semantic_html,10,L30H4; L27H27; L28H1; L27H13; L10H6; L27H9; L1...,L28H28,0.001285,lexical,robustness


In [17]:
cdf = pd.read_csv(PROJECT_ROOT / 'results/adhoc/d6_multihead_ablation/pythia-2.8b-candidate-heads.csv')
cols = ['layer','head','binding_score','own_score','max_other_score',
        'max_other_domain','selective','sink','structural','bos_attention',
        'attn_to_pos1','in_lexical_set']
print(cdf[cdf.compound=='semantic_html'][cols].to_string())

    layer  head  binding_score  own_score  max_other_score max_other_domain  selective   sink  structural  bos_attention  attn_to_pos1  in_lexical_set
54     30    29         0.9920     0.9941           0.7814          general      False  False       False         0.0061        0.0001           False
55     30     4         0.9790     0.9899           0.5664          general       True  False       False         0.0051        0.0001            True
56     27    27         0.9392     0.9458           0.1773            legal       True  False       False         0.0210        0.0032            True
57     28     1         0.9239     0.9607           0.3557          general       True  False       False         0.0570        0.0069            True
58     29    13         0.9223     0.9631           0.8360          general      False  False       False         0.0419        0.0028           False
59     27    13         0.8792     0.8489           0.0963          general       True  False 

In [16]:
# Cell 3: Ledger peek — which compounds exist so far, and the verdict shape
from pathlib import Path
import pandas as pd
led = PROJECT_ROOT / 'results/adhoc/d6_multihead_ablation/pythia-2.8b-multihead-ablation.csv'
if led.exists():
    df = pd.read_csv(led)
    print(df.groupby(['compound', 'role', 'segment'])['kl']
            .agg(['count', 'max']).round(6))
else:
    print('no ledger yet')
cand = PROJECT_ROOT / 'results/adhoc/d6_multihead_ablation/pythia-2.8b-candidate-heads.csv'
if cand.exists():
    cdf = pd.read_csv(cand)
    print('\nearned lexical sets:',
          cdf[cdf['in_lexical_set']].groupby('compound')
             .apply(lambda g: ', '.join(f"L{r.layer}H{r.head}"
                                        for r in g.itertuples()),
                    include_groups=False).to_dict())

                                           count       max
compound      role        segment                         
alt_text      robustness  lexical              3  0.000267
                          structural_tail      5  0.000265
bicycle_wheel neg_control lexical              2  0.000032
                          structural_tail      5  0.000940
screen_reader primary     lexical              2  0.000022
                          structural_tail      5  0.000074
semantic_html robustness  lexical             13  0.001471
stock_market  robustness  lexical              3  0.000285
                          structural_tail      5  0.001665

earned lexical sets: {'alt_text': 'L21H6, L10H26, L12H21', 'screen_reader': 'L29H7, L27H10', 'semantic_html': 'L30H4, L27H27, L28H1, L27H13, L10H6, L27H9, L10H26, L14H12, L23H12, L28H28, L21H15, L23H22, L18H9', 'stock_market': 'L27H24, L18H14, L21H24'}


In [18]:
for p in ["Semantic HTML helps", "A semantic HTML is",
          "A screen reader is", "The purpose of alt text is",
          "A stock market is"]:
    print(f"{p!r:35s} → {model.to_str_tokens(p)}")

'Semantic HTML helps'               → ['<|endoftext|>', 'Sem', 'antic', ' HTML', ' helps']
'A semantic HTML is'                → ['<|endoftext|>', 'A', ' semantic', ' HTML', ' is']
'A screen reader is'                → ['<|endoftext|>', 'A', ' screen', ' reader', ' is']
'The purpose of alt text is'        → ['<|endoftext|>', 'The', ' purpose', ' of', ' alt', ' text', ' is']
'A stock market is'                 → ['<|endoftext|>', 'A', ' stock', ' market', ' is']


## Bring it home

In [19]:
# Cell 4: Zip + download (loss class: ephemeral /content)
import os
os.system('zip -j d6_results.zip '
          'results/adhoc/d6_multihead_ablation/pythia-2.8b-candidate-heads.csv '
          'results/adhoc/d6_multihead_ablation/pythia-2.8b-multihead-ablation.csv')
if IN_COLAB:
    from google.colab import files
    files.download('d6_results.zip')
else:
    print('Local run — results already live in results/adhoc/d6_multihead_ablation/, no zip needed.')

	zip warning: name not matched: results/pythia/pythia-2.8b-candidate-heads.csv
	zip warning: name not matched: results/pythia/pythia-2.8b-multihead-ablation.csv

zip error: Nothing to do! (d6_results.zip)


NameError: name 'IN_COLAB' is not defined